# 📊 13｜辦公室資料：五種提問的整合練習

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/13_office_chart_basics/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：回到前導章節「看懂資料與選對圖表」的辦公室資料，**一個資料集裡有 4 張不同的表**：每換一張表都要重新回答「一列代表什麼」，再依問題選出比較、趨勢、組成、分布、關係五種圖。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：讀入所有工作表、說出各自一列代表什麼，最後用**一張垂直長條圖**回答「各部門全年實際支出多少？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：匯出部門全年支出表（預算、實際、執行率、每人支出）。
3. **選做挑戰（單元五～八）**：趨勢、組成、分布、關係各一題，可以只挑一兩題做。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/辦公室圖表練習資料.xlsx`（工作表「部門月報」「員工問卷」「專案工時」等）與 `data/辦公室採購明細.csv`，來自前導章節的[學生素材](../../看懂資料與選對圖表/學生素材/)，請先閱讀[本例資料卡](./data/DATA_CARD.md)。**每張表的一列都不同**：部門月報＝一部門一個月、員工問卷＝一位員工、專案工時＝一個專案、採購明細＝一筆採購。

### 1. 欄位角色貼標籤
- **部門月報**（一列＝一部門一個月）：🔵 `月份`、`部門`；🟢 `預算（元）`、`實際支出（元）`、`人數`（分母，不可跨月加總）。
- **員工問卷**（一列＝一位員工）：⚪ `員工代號`；🟢 `通勤時間（分鐘）`。
- **專案工時**（一列＝一個專案）：⚪ `專案編號`；🟢 `預估工時`、`實際工時`。
- **採購明細**（一列＝一筆採購）：🔵 `日期`、`類別`；🟢 `金額（元）`。

### 2. 五大提問積木與核心問題
- 🎯 **比較（必做）**：各部門 2025 年全年實際支出多少？哪個部門最高？
- 📈 **趨勢（選做）**：全公司每月實際支出如何變化？
- 🧩 **組成（選做）**：2025 年採購金額中，各類別各占多少？
- 📊 **分布（選做）**：員工通勤時間大多落在幾分鐘？
- 🔗 **關係（選做）**：專案的預估工時與實際工時是否同步？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **60 列不是 60 個部門**：部門月報是 5 個部門 × 12 個月；全年支出要依部門加總。
- ⚠️ **人數不能加總**：每月人數相同，12 個月加總會得到 12 倍；每人支出＝全年支出 ÷ 人數。
- ⚠️ **不同工作表不能直接混用**：問卷、專案、月報的一列意義不同，沒有共同鑰匙前不能合併。
- ⚠️ **組成要先確認期間與分母**：採購明細有 4 筆 2024-12-31 的資料；算 2025 年占比要先排除，並寫明分母。
- ⚠️ **關係不推因果、資料不代表真實組織**：預估與實際工時高度相關，不代表「估得準造成做得快」；本資料為教學用合成資料。

### 4. 決定最適圖表
- **5 個部門比全年支出** ➔ **垂直長條圖**（Y 軸從 0 開始，單位元）。
- **12 個月走勢（選做）** ➔ **折線圖**（X 軸月份由早到晚）。
- **5 個採購類別占比（選做）** ➔ **百分比堆疊長條圖**（類別超過 4 個，不用圓餅圖；寫明分母）。
- **通勤時間分布（選做）** ➔ **直方圖**（加平均與中位數線）。
- **預估 vs 實際工時（選做）** ➔ **散佈圖**（點不連線，加上「預估＝實際」參考線）。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    '辦公室圖表練習資料.xlsx': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E8%BE%A6%E5%85%AC%E5%AE%A4%E5%9C%96%E8%A1%A8%E7%B7%B4%E7%BF%92%E8%B3%87%E6%96%99.xlsx',
    '辦公室採購明細.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E8%BE%A6%E5%85%AC%E5%AE%A4%E6%8E%A1%E8%B3%BC%E6%98%8E%E7%B4%B0.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取所有工作表，說出每張表一列代表什麼（貼標籤）

> 🎯 **思維重點**：一個 Excel 有多張工作表、每張表的觀測單位都不同。先全部列出來，再逐張說出「一列代表什麼」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/辦公室圖表練習資料.xlsx 的「所有」工作表，以及 data/辦公室採購明細.csv：
1. 列出 Excel 中所有工作表名稱。
2. 針對「部門月報」「員工問卷」「專案工時」與採購明細，各顯示前 3 列、列數與欄位名稱。
3. 用文字說明每張表「一列代表什麼」；部門月報有幾個部門、幾個月份？
4. 逐張標記欄位角色：標識、條件、指標。
先不要合併，也不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

all_sheets = pd.read_excel('data/辦公室圖表練習資料.xlsx', sheet_name=None)
print("Excel 工作表：", list(all_sheets.keys()))
monthly = all_sheets['部門月報']
survey = all_sheets['員工問卷']
projects = all_sheets['專案工時']
purchases = pd.read_csv('data/辦公室採購明細.csv')

for name, table, unit in [('部門月報', monthly, '一個部門的一個月'), ('員工問卷', survey, '一位員工'),
                          ('專案工時', projects, '一個專案'), ('採購明細', purchases, '一筆採購')]:
    print(f"\n=== {name}：{len(table)} 列，一列代表「{unit}」 ===")
    display(table.head(3))

print(f"\n部門月報：{monthly['部門'].nunique()} 個部門 × {monthly['月份'].nunique()} 個月 = {len(monthly)} 列（不是 {len(monthly)} 個部門）")

### 📝 單元二【必做】：邊界檢核：依部門加總，人數不能跨月加總

> 🎯 **思維重點**：全年支出要「依部門加總 12 個月」；人數每月都一樣，是分母不是可加總的量。順便計算執行率（實際 ÷ 預算）與每人支出。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用「部門月報」整理各部門 2025 年全年數字：
1. 確認每個部門都有 12 個月份、每個部門的人數是否每月相同。
2. 依部門加總預算與實際支出；人數取每月的值（不要加總），並在註解說明原因。
3. 計算：差異＝實際 − 預算、預算執行率＝實際 ÷ 預算、每人全年支出＝實際 ÷ 人數。
4. 整理成表格，依全年實際支出由高到低排序；金額加千分位，執行率用百分比。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
print("各部門月份數：", monthly.groupby('部門')['月份'].nunique().to_dict())
print("各部門人數每月相同：", (monthly.groupby('部門')['人數'].nunique() == 1).all())

# 預算、支出可以 12 個月加總；人數每月都一樣，是「分母」，取一個月的值即可（加總會變 12 倍）
dept_df = monthly.groupby('部門').agg(人數=('人數', 'first'), 全年預算=('預算（元）', 'sum'), 全年實際支出=('實際支出（元）', 'sum'))
dept_df['差異'] = dept_df['全年實際支出'] - dept_df['全年預算']
dept_df['預算執行率'] = dept_df['全年實際支出'] / dept_df['全年預算']
dept_df['每人全年支出'] = (dept_df['全年實際支出'] / dept_df['人數']).round(0)
dept_df = dept_df.sort_values('全年實際支出', ascending=False)
display(dept_df.style.format({'全年預算': '{:,.0f}', '全年實際支出': '{:,.0f}', '差異': '{:,.0f}',
                              '預算執行率': '{:.1%}', '每人全年支出': '{:,.0f}'}))

### 📝 單元三【必做】：提問積木【比較】— 各部門全年實際支出多少？（垂直長條圖）

> 🎯 **思維重點**：5 個部門比高低，部門名稱短，用【垂直長條圖】；Y 軸從 0 開始、單位元，最高的部門用醒目顏色。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用上一步的表畫出各部門「全年實際支出」的垂直長條圖：
1. X 軸為部門，Y 軸為全年實際支出（元），從 0 開始，依金額由高到低排列。
2. 柱子上方標註金額（千分位）。
3. 全年實際支出最高的部門用醒目顏色標示，寫在標題中。
4. 圖表標題、座標軸全部使用繁體中文，並註明資料期間 2025 年。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

spend = dept_df['全年實際支出']
top_dept = spend.idxmax()
colors = ['#e74c3c' if d == top_dept else '#3498db' for d in spend.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(spend.index, spend.values, color=colors, width=0.55)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40000, f'{bar.get_height():,.0f}', ha='center')
plt.ylim(0, spend.max() * 1.15)
plt.title(f'2025 年各部門全年實際支出（最高：{top_dept}）', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('部門')
plt.ylabel('全年實際支出（元）')
plt.ticklabel_format(axis='y', style='plain')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能說出部門月報、員工問卷、專案工時、採購明細四張表「一列代表什麼」，並說明 60 列不是 60 個部門。
- [ ] 能展示依部門加總後只剩 5 列，長條圖與表格的全年實際支出一致；圖的數值軸從 0 開始。
- [ ] 能用一句話解釋：為什麼「人數」不能和支出一樣 12 個月加總？
- 💬 **討論題**：資訊部門支出最高，但人數也最多（46 人）；改看「每人全年支出」，排名會怎麼變？（答案在單元四）

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 部門月報 60 列＝5 部門 × 12 個月；依部門加總後剩 5 列。
- 長條圖與表格都顯示**資訊**部門全年實際支出最高（3,277,400 元）、**行政**最低（2,108,800 元）。

</details>

若今天繼續做報表，請進行**單元四**；單元五～八各練一種提問，可以只挑一兩題。

---

### 📝 單元四【報表應用】：匯出部門全年支出表

> 🎯 **思維重點**：報表同時放「總額」與「每人支出」：總額回答「誰花最多」，每人支出回答「規模相同時誰花得多」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請將部門全年支出表匯出成 Excel：
1. 欄位：部門、人數、全年預算、全年實際支出、差異、預算執行率、每人全年支出。
2. 存為 output/部門全年支出.xlsx（若 output 目錄不存在請自動建立）。
3. 匯出後重新讀取，核對全年實際支出合計等於部門月報實際支出的總和。
4. 用一句話說明：依「每人全年支出」排序，第一名還是同一個部門嗎？
5. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
os.makedirs('output', exist_ok=True)
output_path = 'output/部門全年支出.xlsx'
dept_df.reset_index().to_excel(output_path, index=False)

check = pd.read_excel(output_path)
print(f"重新讀取全年實際支出合計：{check['全年實際支出'].sum():,}；部門月報總和：{monthly['實際支出（元）'].sum():,}")
print(f"全年實際支出最高：{dept_df['全年實際支出'].idxmax()}；每人全年支出最高：{dept_df['每人全年支出'].idxmax()}")

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 報表！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【趨勢】— 全公司每月實際支出（折線圖）

> 🎯 **思維重點**：「時間 ＋ 指標」看變化，用【折線圖】；先依月份加總 5 個部門。各月差距只有約 1 成，Y 軸從 0 開始可以避免誇大起伏。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用部門月報分析全公司每月實際支出：
1. 依月份加總 5 個部門的實際支出，月份由早到晚排序。
2. 畫折線圖：X 軸為 1月～12月，Y 軸為實際支出（元），從 0 開始，每個點加上圓點；標出最高與最低的月份。
3. 用文字說明：最高與最低月份差多少（金額與百分比）？只有一年資料，能不能說有季節性？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
company = monthly.groupby('月份')['實際支出（元）'].sum()
labels = [f'{m.month}月' for m in company.index]
hi, lo = company.idxmax(), company.idxmin()

plt.figure(figsize=(10, 5))
plt.plot(labels, company.values, marker='o', color='#3498db', linewidth=2)
for point, color in [(hi, '#e74c3c'), (lo, '#2c3e50')]:
    plt.annotate(f'{company[point]:,.0f}', (f'{point.month}月', company[point]), textcoords='offset points',
                 xytext=(0, 10), ha='center', color=color, fontweight='bold')
plt.ylim(0, company.max() * 1.2)
plt.ticklabel_format(axis='y', style='plain')
plt.title('2025 年全公司每月實際支出', fontsize=13, fontweight='bold')
plt.xlabel('月份')
plt.ylabel('實際支出（元）')
plt.grid(linestyle='--', alpha=0.4)
plt.show()
gap = company[hi] - company[lo]
print(f"💡 最高 {hi.month} 月、最低 {lo.month} 月，相差 {gap:,.0f} 元（約 {gap / company[lo]:.1%}）。")
print("⚠️ 邊界提醒：只有一年資料，不能確定這是每年都會重複的季節性。")

### 📝 單元六【選做挑戰】：提問積木【組成】— 2025 年採購金額的類別占比（百分比堆疊長條圖）

> 🎯 **思維重點**：組成要先確認**期間**與**分母**：採購明細有 4 筆 2024-12-31 的資料，排除後才是 2025 年；分母是 2025 年採購總額。類別有 5 個，用百分比堆疊長條圖，不用圓餅圖。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請分析 2025 年採購金額的類別組成：
1. 把採購明細的日期轉成日期格式，列出不是 2025 年的紀錄並排除，印出排除前後筆數。
2. 依類別加總 2025 年的金額，計算占 2025 年採購總額的百分比（保留一位小數），由大到小排序。
3. 畫一根水平的百分比堆疊長條圖（總和 100%），每段標示類別與百分比；標題寫明「分母＝2025 年採購總額 X 元」。
4. 說明為什麼這裡不用圓餅圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
purchases['日期'] = pd.to_datetime(purchases['日期'])
not_2025 = purchases[purchases['日期'].dt.year != 2025]
print("不屬於 2025 年的紀錄：")
display(not_2025)
p2025 = purchases[purchases['日期'].dt.year == 2025]
print(f"排除前 {len(purchases)} 筆 → 2025 年 {len(p2025)} 筆")

share = p2025.groupby('類別')['金額（元）'].sum().sort_values(ascending=False)
total = share.sum()
share_pct = share / total * 100
display(pd.DataFrame({'金額（元）': share, '占比(%)': share_pct.round(1)}))

fig, ax = plt.subplots(figsize=(11, 2.6))
left = 0
for (cat, pct), color in zip(share_pct.items(), plt.cm.Set2.colors):
    ax.barh(0, pct, left=left, color=color)
    ax.text(left + pct / 2, 0, f'{cat}\n{pct:.1f}%', ha='center', va='center', fontsize=10)
    left += pct
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xlabel('占 2025 年採購總額（%）')
ax.set_title(f'2025 年採購金額類別組成（分母＝2025 年採購總額 {total:,.0f} 元）', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 類別有 5 個且占比接近，圓餅圖很難分辨大小；百分比長條圖更容易比較。")

### 📝 單元七【選做挑戰】：提問積木【分布】— 員工通勤時間大多多久？（直方圖）

> 🎯 **思維重點**：單一指標看集中區間，用【直方圖】；一列是一位員工，員工代號只能計數。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用「員工問卷」分析通勤時間：
1. 計算通勤時間（分鐘）的描述統計（人數、平均、中位數、四分位數、最小、最大）。
2. 畫直方圖：X 軸為通勤時間（分鐘，每 10 分鐘一組），Y 軸為人數，加上平均線與中位數線。
3. 標題寫出人數 n，用數字描述通勤時間主要集中的區間。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
commute = survey['通勤時間（分鐘）']
display(commute.describe().round(1))

plt.figure(figsize=(8, 4.5))
plt.hist(commute, bins=range(10, 100, 10), color='#3498db', edgecolor='white')
plt.axvline(commute.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'平均 {commute.mean():.1f} 分鐘')
plt.axvline(commute.median(), color='#2c3e50', linestyle=':', linewidth=2, label=f'中位數 {commute.median():.1f} 分鐘')
plt.title(f'員工通勤時間分布（n = {len(commute)}）', fontsize=13, fontweight='bold')
plt.xlabel('通勤時間（分鐘，每 10 分鐘一組）')
plt.ylabel('人數')
plt.legend()
plt.show()
q1, q3 = commute.quantile([0.25, 0.75])
print(f"💡 一半的員工通勤時間在 {q1:.0f}～{q3:.0f} 分鐘之間。")

### 📝 單元八【選做挑戰】：提問積木【關係】— 預估工時與實際工時同步嗎？（散佈圖）

> 🎯 **思維重點**：兩個成對數值看連動，用【散佈圖】（點不連線）；加一條「預估＝實際」的參考線，線上方的點代表實際做得比預估久。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用「專案工時」分析預估工時與實際工時的關係：
1. 畫散佈圖：X 軸為預估工時、Y 軸為實際工時（小時），每個點代表一個專案，點不要連線。
2. 加上一條「預估＝實際」的虛線參考線。
3. 計算皮爾森相關係數並顯示在標題；統計有幾個專案實際工時超過預估。
4. 用文字解讀關聯強弱，並附註：相關不等於因果，也不能用來評價個人能力。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
corr = projects['預估工時'].corr(projects['實際工時'])
over = (projects['實際工時'] > projects['預估工時']).sum()

plt.figure(figsize=(7, 6))
plt.scatter(projects['預估工時'], projects['實際工時'], color='#2ecc71', edgecolors='#27ae60', s=55, alpha=0.8)
lim = [0, max(projects['預估工時'].max(), projects['實際工時'].max()) * 1.1]
plt.plot(lim, lim, color='gray', linestyle='--', label='預估 = 實際')
plt.xlim(lim)
plt.ylim(lim)
plt.title(f'專案預估工時與實際工時（r = {corr:.2f}，n = {len(projects)}）', fontsize=12, fontweight='bold')
plt.xlabel('預估工時（小時）')
plt.ylabel('實際工時（小時）')
plt.legend()
plt.grid(linestyle='--', alpha=0.4)
plt.show()
print(f"💡 相關係數 r = {corr:.2f}，兩者高度正相關；{over} / {len(projects)} 個專案實際工時超過預估。")
print("⚠️ 邊界提醒：相關不等於因果，也不能用這張圖評價個人能力；本資料為教學用合成資料。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：部門月報依部門加總為 5 列；資訊部門全年實際支出最高，長條圖從 0 開始。
- [ ] **報表應用**：Excel 含預算執行率與每人全年支出，合計與月報一致。
- [ ] **選做挑戰**：趨勢、組成、分布、關係四張圖各自說得出資料表、分母與選圖理由。
- [ ] **AI 協作**：能說明自己提出的問題、AI 幫忙執行的事，以及自己如何確認結果。